# Łączenie danych w pandas — `merge`, `join`, `concat`, `merge_asof`, `merge_ordered`, `combine_first`, `combine`, `update`

Notebook referencyjny. Sygnatury i zachowania zweryfikowane na **pandas 3.0.2**.

**Legenda w tabelach parametrów**

| Znacznik | Znaczenie |
|---|---|
| 🔴 | parametr **wymagany** (brak wartości domyślnej) |
| 🟢 | parametr **opcjonalny** (ma wartość domyślną — podana w kolumnie „Domyślnie") |

**Konwencja:** kod jest pisany bezpośrednio w komórkach (bez opakowywania w funkcje), żeby łatwo było go skopiować i zmodyfikować. Testowe `try/except` służą tylko do pokazania komunikatów błędów.

**Co zmieniło się w pandas 3.0, a ma znaczenie przy łączeniu danych**

- `copy=` jest w `merge` / `concat` **deprecated i ignorowany** — od 3.0 obowiązuje Copy-on-Write (nie używaj tego parametru).
- Domyślny dtype tekstu to `str` (zamiast `object`) — klucze tekstowe łączą się po nowym dtype.
- `how="left_anti"` / `"right_anti"` są dostępne bezpośrednio w `merge` (w starszych wersjach: `indicator=True` + filtr) — zwróć uwagę na zachowanie opisane w sekcji 2.3.
- Rozdzielczość czasu w `datetime64` bywa różna (`us` vs `ns`) — istotne dla `merge_asof` (sekcja 5.8).

## Spis treści
1. Mapa decyzyjna — którego narzędzia użyć
2. `pd.merge` / `DataFrame.merge` — JOIN po kluczu
3. `DataFrame.join` — JOIN po indeksie
4. `pd.concat` — sklejanie (wiersze / kolumny)
5. `pd.merge_asof` — dopasowanie do **najbliższego** klucza
6. `pd.merge_ordered` — uporządkowane łączenie szeregów z wypełnianiem
7. Łączenie częściowo nakładających się danych: `combine_first`, `combine`, `update`
8. Wydajność, dobre praktyki, alternatywy (Polars, DuckDB, SQL Server, Power Query)
9. Ściąga

## 1. Mapa decyzyjna

| Potrzeba | Narzędzie | Dopasowanie po | Relacja wierszy |
|---|---|---|---|
| Klasyczny JOIN jak w SQL | `merge` | kluczu (kolumny/indeks) | 1:1, 1:n, n:m (mnoży wiersze) |
| Dołączenie kolumn po indeksie, wiele ramek naraz | `join` | indeksie | jak `merge` (`how="left"` domyślnie) |
| Dopisanie wierszy pod spodem / kolumn obok siebie | `concat` | pozycji / etykietach osi | brak „klucza" — tylko wyrównanie osi |
| Klucz „najbliższy" (kursy walut, cenniki, zdarzenia w czasie) | `merge_asof` | najbliższym kluczu (≤, ≥ lub nearest) | **max 1 dopasowanie** na wiersz lewej ramki |
| Scalenie dwóch uporządkowanych szeregów + wypełnienie luk | `merge_ordered` | kluczu, wynik posortowany | jak outer join + opcjonalny `ffill` |
| Uzupełnienie braków w A wartościami z B | `combine_first` | **etykietach** indeksu i kolumn | union indeksów/kolumn |
| Własna reguła wyboru wartości (min, max, priorytet…) | `combine` | **etykietach** indeksu i kolumn | union indeksów/kolumn, func per kolumna |
| Nadpisanie wartości w miejscu (in-place) | `update` | **etykietach** indeksu i kolumn | tylko wiersze/kolumny ramki wywołującej |

**Trzy pytania, które rozstrzygają wybór**

1. **Po czym dopasowuję?** klucz w kolumnie → `merge`; indeks → `join`; brak klucza, tylko układ osi → `concat`; „najbliższy" → `merge_asof`.
2. **Jaka relacja?** Jeśli klucz nie jest unikalny po jednej ze stron, `merge` mnoży wiersze — kontroluj to przez `validate=`.
3. **Co z niedopasowanymi?** `how=` (`inner`/`left`/`right`/`outer`/`*_anti`) albo `indicator=True` do audytu.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
print(pd.__version__)

3.0.2


### Dane przykładowe (używane w sekcjach 2–3)

Celowo zawierają: klienta bez zamówień (Dawid, `4`) oraz zamówienie z nieistniejącym klientem (`5`) — to pozwala zobaczyć różnicę między typami `how`.

In [2]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Anna", "Bartek", "Celina", "Dawid"],
    "city": ["Bydgoszcz", "Toruń", "Gdańsk", "Poznań"],
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 1, 2, 3, 5],
    "amount": [250.0, 120.0, 80.0, 300.0, 45.0],
})

display(customers)
display(orders)

,customer_id,name,city
0,1,Anna,Bydgoszcz
1,2,Bartek,Toruń
2,3,Celina,Gdańsk
3,4,Dawid,Poznań


,order_id,customer_id,amount
0,101,1,250.0
1,102,1,120.0
2,103,2,80.0
3,104,3,300.0
4,105,5,45.0


## 2. `pd.merge` / `DataFrame.merge`

```python
pd.merge(left, right, how="inner", on=None, left_on=None, right_on=None,
         left_index=False, right_index=False, sort=False,
         suffixes=("_x", "_y"), copy=<no_default>, indicator=False, validate=None)

DataFrame.merge(right, how="inner", on=None, ...)   # to samo, left = self
```

`left.merge(right, ...)` i `pd.merge(left, right, ...)` są równoważne — metoda jest tylko wygodniejsza w łańcuchach (`.pipe`, `.assign`, …).

### Parametry

| Parametr | Wymagany? | Domyślnie | Opis |
|---|---|---|---|
| `left` | 🔴 (tylko `pd.merge`; w metodzie to `self`) | — | Lewa ramka (lub `Series` z nazwą). |
| `right` | 🔴 | — | Prawa ramka (lub `Series` z nazwą). |
| `how` | 🟢 | `"inner"` | Typ złączenia: `inner`, `left`, `right`, `outer` (pełne), `cross` (iloczyn kartezjański), `left_anti`, `right_anti` (zob. 2.3). Kolejność wyniku: `inner`/`left`/`right` zachowują kolejność kluczy strony „głównej", `outer` **sortuje klucze leksykograficznie**, `cross` zachowuje kolejność lewej. |
| `on` | 🟢 | `None` | Nazwa(y) kolumn **lub poziomów indeksu** występujące w obu ramkach. Gdy `None` i brak `left_index`/`right_index` → **przecięcie nazw kolumn** (niejawne — unikaj w kodzie produkcyjnym). |
| `left_on` | 🟢 | `None` | Klucz(e) w lewej ramce, gdy nazwy się różnią. Może to być też tablica o długości ramki. Nie łączyć z `on`. |
| `right_on` | 🟢 | `None` | Klucz(e) w prawej ramce. Liczba kluczy musi odpowiadać `left_on`. |
| `left_index` | 🟢 | `False` | Użyj indeksu lewej ramki jako klucza. |
| `right_index` | 🟢 | `False` | Użyj indeksu prawej ramki jako klucza. |
| `sort` | 🟢 | `False` | Sortuj wynik po kluczach złączenia. Dla `how="outer"` klucze i tak są sortowane. `sort=False` bywa szybsze. |
| `suffixes` | 🟢 | `("_x", "_y")` | Sufiksy dla **kolidujących nazw kolumn niebędących kluczem**. Jedna z wartości może być `None` (kolumna bez sufiksu), ale nie obie. |
| `copy` | 🟢 | *(brak)* | **Deprecated w 3.0, ignorowany** (Copy-on-Write). Nie używaj. |
| `indicator` | 🟢 | `False` | `True` → dodaje kolumnę `_merge` (`left_only` / `right_only` / `both`); `str` → własna nazwa kolumny. Narzędzie audytu jakości złączenia. |
| `validate` | 🟢 | `None` | Sprawdza kardynalność kluczy i rzuca `MergeError`: `"one_to_one"`/`"1:1"`, `"one_to_many"`/`"1:m"`, `"many_to_one"`/`"m:1"`, `"many_to_many"`/`"m:m"` (ten ostatni niczego nie sprawdza). |

### 2.1 Podstawowy join (`on`)

In [3]:
# on -- klucz o tej samej nazwie w obu ramkach; how domyślnie "inner"
pd.merge(customers, orders, on="customer_id")
# równoważnie: customers.merge(orders, on="customer_id")

,customer_id,name,city,order_id,amount
0,1,Anna,Bydgoszcz,101,250.0
1,1,Anna,Bydgoszcz,102,120.0
2,2,Bartek,Toruń,103,80.0
3,3,Celina,Gdańsk,104,300.0


### 2.2 Typy złączenia (`how`)

- `inner` — tylko klucze obecne w obu ramkach,
- `left` — wszystkie wiersze lewej, brakujące dopasowania → `NaN`,
- `right` — analogicznie dla prawej,
- `outer` — suma kluczy (klucze posortowane).

In [4]:
for how in ["inner", "left", "right", "outer"]:
    print(f"how={how!r}")
    display(customers.merge(orders, on="customer_id", how=how))

how='inner'


,customer_id,name,city,order_id,amount
0,1,Anna,Bydgoszcz,101,250.0
1,1,Anna,Bydgoszcz,102,120.0
2,2,Bartek,Toruń,103,80.0
3,3,Celina,Gdańsk,104,300.0


how='left'


,customer_id,name,city,order_id,amount
0,1,Anna,Bydgoszcz,101.0,250.0
1,1,Anna,Bydgoszcz,102.0,120.0
2,2,Bartek,Toruń,103.0,80.0
3,3,Celina,Gdańsk,104.0,300.0
4,4,Dawid,Poznań,NaN,NaN


how='right'


,customer_id,name,city,order_id,amount
0,1,Anna,Bydgoszcz,101,250.0
1,1,Anna,Bydgoszcz,102,120.0
2,2,Bartek,Toruń,103,80.0
3,3,Celina,Gdańsk,104,300.0
4,5,NaN,NaN,105,45.0


how='outer'


,customer_id,name,city,order_id,amount
0,1,Anna,Bydgoszcz,101.0,250.0
1,1,Anna,Bydgoszcz,102.0,120.0
2,2,Bartek,Toruń,103.0,80.0
3,3,Celina,Gdańsk,104.0,300.0
4,4,Dawid,Poznań,NaN,NaN
5,5,NaN,NaN,105.0,45.0


### 2.3 Anti-join i semi-join

Jeśli chcesz tylko **przefiltrować** lewą ramkę (nie potrzebujesz kolumn z prawej), `merge` nie jest najlepszym narzędziem:

- **Anti-join** (klienci **bez** zamówień) i **semi-join** (klienci **z** zamówieniami): `isin` / `~isin` — najprostsze, najszybsze, **nie mnoży wierszy** i nie zmienia kolumn.
- `how="left_anti"` / `how="right_anti"` — dostępne w pandas 3.0.2 (sprawdź `help(pd.merge)` w swojej wersji). ⚠️ W tej wersji wynik **zawiera także kolumny prawej ramki wypełnione `NaN`** i zachowuje „dziwny" indeks — zwykle trzeba zawęzić kolumny i zrobić `reset_index(drop=True)`.
- `indicator=True` + filtr `_merge == "left_only"` — działa we wszystkich wersjach i jest przydatne, gdy audyt i tak jest potrzebny.

In [5]:
has_orders = customers["customer_id"].isin(orders["customer_id"])

print("anti-join przez ~isin (klienci BEZ zamówień):")
display(customers[~has_orders])

print("semi-join przez isin (klienci Z zamówieniami, bez mnożenia wierszy):")
display(customers[has_orders])

print("how='left_anti' -- uwaga na dodatkowe kolumny prawej ramki (NaN) i indeks:")
display(customers.merge(orders, on="customer_id", how="left_anti"))

print("po zawężeniu do kolumn lewej ramki:")
display(customers.merge(orders, on="customer_id", how="left_anti")[customers.columns].reset_index(drop=True))

print("indicator -- gdy potrzebny jest też audyt:")
m = customers.merge(orders[["customer_id"]].drop_duplicates(), on="customer_id",
                    how="left", indicator=True)
display(m.loc[m["_merge"] == "left_only"].drop(columns="_merge"))

anti-join przez ~isin (klienci BEZ zamówień):


,customer_id,name,city
3,4,Dawid,Poznań


semi-join przez isin (klienci Z zamówieniami, bez mnożenia wierszy):


,customer_id,name,city
0,1,Anna,Bydgoszcz
1,2,Bartek,Toruń
2,3,Celina,Gdańsk


how='left_anti' -- uwaga na dodatkowe kolumny prawej ramki (NaN) i indeks:


,customer_id,name,city,order_id,amount
4,4,Dawid,Poznań,NaN,NaN


po zawężeniu do kolumn lewej ramki:


,customer_id,name,city
0,4,Dawid,Poznań


indicator -- gdy potrzebny jest też audyt:


,customer_id,name,city
3,4,Dawid,Poznań


### 2.4 `indicator` — audyt jakości złączenia

Kolumna wskaźnikowa ma dtype `category`. W pipeline’ach warto po złączeniu asercją sprawdzić, że nie ma „osieroconych" rekordów.

In [6]:
audit = customers.merge(orders, on="customer_id", how="outer", indicator="zrodlo")
display(audit)
print(audit["zrodlo"].value_counts())

# asercja jakości danych: każde zamówienie musi mieć klienta
n_orphans = (audit["zrodlo"] == "right_only").sum()
print("zamówienia bez klienta:", n_orphans)

,customer_id,name,city,order_id,amount,zrodlo
0,1,Anna,Bydgoszcz,101.0,250.0,both
1,1,Anna,Bydgoszcz,102.0,120.0,both
2,2,Bartek,Toruń,103.0,80.0,both
3,3,Celina,Gdańsk,104.0,300.0,both
4,4,Dawid,Poznań,NaN,NaN,left_only
5,5,NaN,NaN,105.0,45.0,right_only


zrodlo
both          4
left_only     1
right_only    1
Name: count, dtype: int64
zamówienia bez klienta: 1


### 2.5 `left_on` / `right_on` — różne nazwy kluczy

Obie kolumny kluczowe **zostają** w wyniku (przy `on=` byłaby jedna) — zwykle jedną trzeba usunąć.

In [7]:
orders_renamed = orders.rename(columns={"customer_id": "cust_id"})

customers.merge(orders_renamed, left_on="customer_id", right_on="cust_id").drop(columns="cust_id")

,customer_id,name,city,order_id,amount
0,1,Anna,Bydgoszcz,101,250.0
1,1,Anna,Bydgoszcz,102,120.0
2,2,Bartek,Toruń,103,80.0
3,3,Celina,Gdańsk,104,300.0


### 2.6 Klucze złożone (lista kolumn)

`on=["a", "b"]` łączy po **parze** wartości. Analogicznie `left_on=[...]`, `right_on=[...]` — kolejność i liczba kluczy muszą się zgadzać.

In [8]:
sales = pd.DataFrame({"year": [2025, 2025, 2026], "region": ["N", "S", "N"], "sales": [10, 20, 30]})
plan  = pd.DataFrame({"yr":   [2025, 2025, 2026], "reg":    ["N", "S", "N"], "plan":  [12, 18, 35]})

sales.merge(plan, left_on=["year", "region"], right_on=["yr", "reg"]).drop(columns=["yr", "reg"])

,year,region,sales,plan
0,2025,N,10,12
1,2025,S,20,18
2,2026,N,30,35


### 2.7 Indeks jako klucz (`left_index` / `right_index`) oraz `on` z nazwą poziomu indeksu

`on` akceptuje także **nazwę poziomu indeksu** — dzięki temu nie trzeba `reset_index()`.

In [9]:
cust_idx = customers.set_index("customer_id")   # indeks nazwany "customer_id"

print("indeks lewej vs kolumna prawej (left_index + right_on) -- indeks wyniku pochodzi z prawej ramki:")
display(cust_idx.merge(orders, left_index=True, right_on="customer_id"))

print("nazwa poziomu indeksu w on (kolumna klucza wraca jako zwykła kolumna, indeks -> RangeIndex):")
display(cust_idx.merge(orders, on="customer_id"))

print("indeks vs indeks (left_index + right_index) -- indeks (klucz) zachowany:")
display(cust_idx.merge(orders.set_index("customer_id"), left_index=True, right_index=True))

indeks lewej vs kolumna prawej (left_index + right_on) -- indeks wyniku pochodzi z prawej ramki:


,name,city,order_id,customer_id,amount
0,Anna,Bydgoszcz,101,1,250.0
1,Anna,Bydgoszcz,102,1,120.0
2,Bartek,Toruń,103,2,80.0
3,Celina,Gdańsk,104,3,300.0


nazwa poziomu indeksu w on (kolumna klucza wraca jako zwykła kolumna, indeks -> RangeIndex):


,customer_id,name,city,order_id,amount
0,1,Anna,Bydgoszcz,101,250.0
1,1,Anna,Bydgoszcz,102,120.0
2,2,Bartek,Toruń,103,80.0
3,3,Celina,Gdańsk,104,300.0


indeks vs indeks (left_index + right_index) -- indeks (klucz) zachowany:


,name,city,order_id,amount
customer_id,,,,
1,Anna,Bydgoszcz,101,250.0
1,Anna,Bydgoszcz,102,120.0
2,Bartek,Toruń,103,80.0
3,Celina,Gdańsk,104,300.0


### 2.8 `suffixes` — kolidujące kolumny

Kolumny o tej samej nazwie, **które nie są kluczem**, dostają sufiksy. Jeden z sufiksów może być `None` (kolumna z tej strony zachowuje nazwę), ale nie oba.

In [10]:
orders_city = orders.assign(city=["Bydgoszcz", "Bydgoszcz", "Toruń", "Gdańsk", "Kraków"])

print("domyślne sufiksy:")
display(orders_city.merge(customers, on="customer_id"))

print("własne sufiksy:")
display(orders_city.merge(customers, on="customer_id", suffixes=("_zam", "_klient")))

print("jedna strona bez sufiksu:")
display(orders_city.merge(customers, on="customer_id", suffixes=(None, "_klient")))

domyślne sufiksy:


,order_id,customer_id,amount,city_x,name,city_y
0,101,1,250.0,Bydgoszcz,Anna,Bydgoszcz
1,102,1,120.0,Bydgoszcz,Anna,Bydgoszcz
2,103,2,80.0,Toruń,Bartek,Toruń
3,104,3,300.0,Gdańsk,Celina,Gdańsk


własne sufiksy:


,order_id,customer_id,amount,city_zam,name,city_klient
0,101,1,250.0,Bydgoszcz,Anna,Bydgoszcz
1,102,1,120.0,Bydgoszcz,Anna,Bydgoszcz
2,103,2,80.0,Toruń,Bartek,Toruń
3,104,3,300.0,Gdańsk,Celina,Gdańsk


jedna strona bez sufiksu:


,order_id,customer_id,amount,city,name,city_klient
0,101,1,250.0,Bydgoszcz,Anna,Bydgoszcz
1,102,1,120.0,Bydgoszcz,Anna,Bydgoszcz
2,103,2,80.0,Toruń,Bartek,Toruń
3,104,3,300.0,Gdańsk,Celina,Gdańsk


### 2.9 `validate` — kontrola kardynalności

Zabezpiecza przed cichym **mnożeniem wierszy** (najczęstszy błąd przy JOIN-ach). Rzuca `pd.errors.MergeError`.

| Wartość | Wymaga unikalnych kluczy w… |
|---|---|
| `"1:1"` / `"one_to_one"` | lewej **i** prawej |
| `"1:m"` / `"one_to_many"` | lewej |
| `"m:1"` / `"many_to_one"` | prawej |
| `"m:m"` / `"many_to_many"` | nigdzie (brak kontroli) |

In [11]:
# zamówienia -> klienci to relacja wiele-do-jednego: m:1 przechodzi
display(orders.merge(customers, on="customer_id", validate="m:1").shape)

# nieprawidłowa deklaracja: klucz customer_id NIE jest unikalny w orders
try:
    customers.merge(orders, on="customer_id", validate="1:1")
except pd.errors.MergeError as e:
    print("MergeError:", e)

(4, 5)

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  customer_id
           1 ...


### 2.10 Mnożenie wierszy (n:m)

Gdy klucz powtarza się po obu stronach, wynik ma iloczyn liczności per klucz. Klucz `1` występuje 2× w każdej ramce → 4 wiersze.

In [12]:
a = pd.DataFrame({"k": [1, 1, 2], "x": [1, 2, 3]})
b = pd.DataFrame({"k": [1, 1, 2], "y": [10, 20, 30]})

res = a.merge(b, on="k")
print(len(a), "x", len(b), "->", len(res))
res

3 x 3 -> 5


,k,x,y
0,1,1,10
1,1,1,20
2,1,2,10
3,1,2,20
4,2,3,30


### 2.11 `how="cross"` — iloczyn kartezjański

Nie wolno podawać kluczy (`on`, `left_on`, …). Przydatne do generowania siatek kombinacji (np. produkt × miesiąc), potem `left join` faktów.

In [13]:
sizes = pd.DataFrame({"size": ["S", "M"]})
colors = pd.DataFrame({"color": ["red", "blue", "green"]})

sizes.merge(colors, how="cross")

,size,color
0,S,red
1,S,blue
2,S,green
3,M,red
4,M,blue
5,M,green


### 2.12 `sort`

Bez `sort=True` kolejność wyniku zależy od `how`. Jeśli kolejność ma znaczenie, lepiej po złączeniu jawnie użyć `sort_values(...)` niż polegać na `sort=`.

In [14]:
left_unsorted = pd.DataFrame({"k": [3, 1, 2], "a": list("xyz")})
right_unsorted = pd.DataFrame({"k": [2, 3, 1], "b": list("uvw")})

print("sort=False (domyślnie):")
display(left_unsorted.merge(right_unsorted, on="k", how="left"))
print("sort=True:")
display(left_unsorted.merge(right_unsorted, on="k", how="left", sort=True))

sort=False (domyślnie):


,k,a,b
0,3,x,v
1,1,y,w
2,2,z,u


sort=True:


,k,a,b
0,1,y,w
1,2,z,u
2,3,x,v


### 2.13 Pułapki

**1) `NaN` łączy się z `NaN`.** W przeciwieństwie do SQL (`NULL = NULL` → nieprawda), pandas dopasowuje brakujące klucze do siebie. Przed złączeniem odfiltruj klucze puste albo je jawnie obsłuż.

**2) Niezgodne dtypes kluczy** (np. `int` vs `str`) kończą się błędem — typy trzeba ujednolicić przed `merge`.

In [15]:
l2 = pd.DataFrame({"k": [1.0, np.nan], "a": [1, 2]})
r2 = pd.DataFrame({"k": [np.nan, 1.0], "b": [3, 4]})
print("NaN dopasowany do NaN:")
display(l2.merge(r2, on="k"))

print("bez pustych kluczy:")
display(l2.dropna(subset=["k"]).merge(r2.dropna(subset=["k"]), on="k"))

try:
    customers.merge(orders.astype({"customer_id": "str"}), on="customer_id")
except ValueError as e:
    print("ValueError:", e)

NaN dopasowany do NaN:


,k,a,b
0,1.0,1,4
1,NaN,2,3


bez pustych kluczy:


,k,a,b
0,1.0,1,4


ValueError: You are trying to merge on int64 and str columns for key 'customer_id'. If you wish to proceed you should use pd.concat


## 3. `DataFrame.join`

```python
DataFrame.join(other, on=None, how="left", lsuffix="", rsuffix="", sort=False, validate=None)
```

`join` to **skrót do `merge` po indeksie**. Domyślnie łączy **indeks lewej ramki z indeksem prawej**. Nie istnieje odpowiednik `pd.join(...)` — to tylko metoda.

### Parametry

| Parametr | Wymagany? | Domyślnie | Opis |
|---|---|---|---|
| `other` | 🔴 | — | `DataFrame`, nazwana `Series` **lub lista** ramek/serii (lista działa tylko przy łączeniu po indeksie). |
| `on` | 🟢 | `None` | Kolumna/poziom indeksu **w ramce wywołującej** (`self`), którą dopasowujemy do **indeksu** `other`. Przy `None` → indeks do indeksu. Dla listy `other` musi być `None`. |
| `how` | 🟢 | `"left"` | `left`, `right`, `outer`, `inner`, `cross`. ⚠️ Domyślnie **`left`** (w `merge` jest `inner`). |
| `lsuffix` | 🟢 | `""` | Sufiks kolidujących kolumn z lewej ramki. |
| `rsuffix` | 🟢 | `""` | Sufiks kolidujących kolumn z prawej ramki. Przy kolizji nazw i **braku obu sufiksów** — `ValueError`. |
| `sort` | 🟢 | `False` | Sortuje klucze złączenia leksykograficznie. |
| `validate` | 🟢 | `None` | Jak w `merge`: `"1:1"`, `"1:m"`, `"m:1"`, `"m:m"`. |

Różnice względem `merge`: brak `left_on`/`right_on`/`indicator`, inne domyślne `how`, sufiksy w osobnych parametrach (`lsuffix`/`rsuffix`), obsługa **listy** ramek w jednym wywołaniu.

### 3.1 `on` — kolumna lewej ramki vs indeks prawej

In [16]:
cust_idx = customers.set_index("customer_id")

# kolumna customer_id w orders dopasowana do INDEKSU cust_idx; how="left" domyślnie
orders.join(cust_idx, on="customer_id")

,order_id,customer_id,amount,name,city
0,101,1,250.0,Anna,Bydgoszcz
1,102,1,120.0,Anna,Bydgoszcz
2,103,2,80.0,Bartek,Toruń
3,104,3,300.0,Celina,Gdańsk
4,105,5,45.0,NaN,NaN


### 3.2 Indeks do indeksu

Typowy wzorzec: agregat z `groupby` (indeks = klucz grupowania) dołączony z powrotem do tabeli wymiarowej.

In [17]:
revenue = orders.groupby("customer_id")["amount"].agg(total="sum", n_orders="count")
display(revenue)

cust_idx.join(revenue)                    # how="left": Dawid dostaje NaN

,total,n_orders
customer_id,,
1,370.0,2
2,80.0,1
3,300.0,1
5,45.0,1


,name,city,total,n_orders
customer_id,,,,
1,Anna,Bydgoszcz,370.0,2.0
2,Bartek,Toruń,80.0,1.0
3,Celina,Gdańsk,300.0,1.0
4,Dawid,Poznań,NaN,NaN


### 3.3 Wiele ramek naraz (lista `other`)

Jedno wywołanie zamiast łańcucha `join().join().join()`. Wszystkie elementy listy muszą mieć **unikalne nazwy kolumn** (albo być nazwanymi `Series`); `on` musi być `None`.

In [18]:
segment = pd.Series({1: "A", 2: "B", 3: "A"}, name="segment")
segment.index.name = "customer_id"

cust_idx.join([revenue, segment], how="left")

,name,city,total,n_orders,segment
customer_id,,,,,
1,Anna,Bydgoszcz,370.0,2.0,A
2,Bartek,Toruń,80.0,1.0,B
3,Celina,Gdańsk,300.0,1.0,A
4,Dawid,Poznań,NaN,NaN,NaN


### 3.4 `lsuffix` / `rsuffix`

In [19]:
try:
    cust_idx.join(cust_idx)                # kolidujące kolumny name, city, brak sufiksów
except ValueError as e:
    print("ValueError:", e)

cust_idx.join(cust_idx, lsuffix="_l", rsuffix="_r")

ValueError: columns overlap but no suffix specified: Index(['name', 'city'], dtype='str')


,name_l,city_l,name_r,city_r
customer_id,,,,
1,Anna,Bydgoszcz,Anna,Bydgoszcz
2,Bartek,Toruń,Bartek,Toruń
3,Celina,Gdańsk,Celina,Gdańsk
4,Dawid,Poznań,Dawid,Poznań


### `merge` czy `join`?

- Klucz jest w **kolumnach** → `merge` (jawnie, czytelnie).
- Dane mają już **sensowny indeks** (np. wynik `groupby`) lub łączysz **wiele ramek po indeksie** → `join`.
- Nie robi się `set_index` tylko po to, żeby użyć `join` — nie przyspiesza to złączenia, a psuje czytelność.

## 4. `pd.concat`

```python
pd.concat(objs, *, axis=0, join="outer", ignore_index=False, keys=None,
          levels=None, names=None, verify_integrity=False, sort=False, copy=<no_default>)
```

`concat` **nie dopasowuje po kluczu** — wyrównuje obiekty po etykietach **drugiej** osi (tej, wzdłuż której nie sklejamy) i układa je jeden za drugim. Wszystkie parametry poza `objs` są **keyword-only** (`*` w sygnaturze).

### Parametry

| Parametr | Wymagany? | Domyślnie | Opis |
|---|---|---|---|
| `objs` | 🔴 | — | Iterowalny zbiór `DataFrame`/`Series` **albo słownik** (klucze słownika stają się `keys`). Puste `None` są pomijane. |
| `axis` | 🟢 | `0` | `0` / `"index"` — sklejanie wierszy (jedne pod drugimi); `1` / `"columns"` — kolumn (obok siebie). |
| `join` | 🟢 | `"outer"` | Co z etykietami drugiej osi: `"outer"` (suma, braki → `NaN`) lub `"inner"` (przecięcie). |
| `ignore_index` | 🟢 | `False` | `True` → wynik dostaje nowy indeks `0..n-1` (oryginalne etykiety ignorowane). |
| `keys` | 🟢 | `None` | Etykiety identyfikujące źródło każdego obiektu → tworzy **MultiIndex** (poziom zewnętrzny). |
| `levels` | 🟢 | `None` | Jawna lista wartości dla poziomów MultiIndexu (kolejność/kategorie, także nieużyte). Rzadko potrzebne. |
| `names` | 🟢 | `None` | Nazwy poziomów MultiIndexu tworzonego przez `keys`/`levels`. |
| `verify_integrity` | 🟢 | `False` | `True` → `ValueError`, jeśli wynikowa oś sklejania ma **zduplikowane etykiety**. |
| `sort` | 🟢 | `False` | Sortuje etykiety drugiej osi (np. kolumny przy `axis=0`), gdy nie są wyrównane. Wyjątek: `DatetimeIndex` przy `join="outer"` jest sortowany zawsze. |
| `copy` | 🟢 | *(brak)* | **Deprecated w 3.0, ignorowany.** |

### 4.1 `axis=0` — dopisywanie wierszy (domyślnie)

In [20]:
q1 = pd.DataFrame({"id": [1, 2], "val": [10, 20]})
q2 = pd.DataFrame({"id": [3, 4], "val": [30, 40], "extra": ["a", "b"]})

# outer: suma kolumn, brak "extra" w q1 -> NaN; indeks dubluje się (0,1,0,1)
pd.concat([q1, q2])

,id,val,extra
0,1,10,NaN
1,2,20,NaN
0,3,30,a
1,4,40,b


### 4.2 `ignore_index`

In [21]:
pd.concat([q1, q2], ignore_index=True)

,id,val,extra
0,1,10,NaN
1,2,20,NaN
2,3,30,a
3,4,40,b


### 4.3 `join="inner"` — tylko wspólne kolumny

In [22]:
pd.concat([q1, q2], join="inner", ignore_index=True)

,id,val
0,1,10
1,2,20
2,3,30
3,4,40


### 4.4 `keys` + `names` — ślad pochodzenia (MultiIndex)

Bardzo przydatne przy sklejaniu wielu plików/partycji: wiesz, z którego źródła jest wiersz. Potem `reset_index(level=0)` zamienia poziom na zwykłą kolumnę.

In [23]:
res = pd.concat([q1, q2], keys=["Q1", "Q2"], names=["kwartal", None])
display(res)

# poziom zewnętrzny -> zwykła kolumna
res.reset_index(level="kwartal").reset_index(drop=True)

id  val extra
kwartal                 
Q1      0   1   10   NaN
        1   2   20   NaN
Q2      0   3   30     a
        1   4   40     b

,kwartal,id,val,extra
0,Q1,1,10,NaN
1,Q1,2,20,NaN
2,Q2,3,30,a
3,Q2,4,40,b


### 4.5 Słownik zamiast listy

Klucze słownika działają jak `keys=`.

In [24]:
pd.concat({"Q1": q1, "Q2": q2}, names=["kwartal", None])

id  val extra
kwartal                 
Q1      0   1   10   NaN
        1   2   20   NaN
Q2      0   3   30     a
        1   4   40     b

### 4.6 `levels` — jawna definicja poziomów MultiIndexu

Pozwala ustalić kolejność (i pełny zestaw) wartości poziomu — np. gdy chcesz mieć też poziomy, dla których na razie nie ma danych.

In [25]:
res_lv = pd.concat([q1, q2], keys=["Q1", "Q3"], levels=[["Q1", "Q2", "Q3", "Q4"]], names=["kwartal", None])
print(res_lv.index.levels[0])     # wszystkie zdefiniowane poziomy, także nieużyte
print(res_lv.index.remove_unused_levels().levels[0])

Index(['Q1', 'Q2', 'Q3', 'Q4'], dtype='str', name='kwartal')
Index(['Q1', 'Q3'], dtype='str', name='kwartal')


### 4.7 `verify_integrity`

In [26]:
try:
    pd.concat([q1, q2], verify_integrity=True)      # indeksy 0,1 powtarzają się
except ValueError as e:
    print("ValueError:", e)

pd.concat([q1, q2], ignore_index=True, verify_integrity=True)   # ok, indeks nowy

ValueError: Indexes have overlapping values: Index([0, 1], dtype='int64')


,id,val,extra
0,1,10,NaN
1,2,20,NaN
2,3,30,a
3,4,40,b


### 4.8 `sort` — kolejność kolumn

In [27]:
ca = pd.DataFrame({"b": [1], "a": [2]})
cb = pd.DataFrame({"c": [3], "a": [4]})

print("sort=False:", list(pd.concat([ca, cb]).columns))
print("sort=True: ", list(pd.concat([ca, cb], sort=True).columns))

sort=False: ['b', 'a', 'c']
sort=True:  ['a', 'b', 'c']


### 4.9 `axis=1` — dokładanie kolumn (wyrównanie po indeksie)

To **złączenie po indeksie** — z `join="outer"` działa jak `outer join`, z `"inner"` jak `inner join`. Przy `Series` nazwy stają się nazwami kolumn (albo `keys`).

In [28]:
s1 = pd.Series([1, 2, 3], index=list("abc"), name="x")
s2 = pd.Series([10, 20], index=list("bd"), name="y")

print("outer:");  display(pd.concat([s1, s2], axis=1))
print("inner:");  display(pd.concat([s1, s2], axis=1, join="inner"))
print("keys:");   display(pd.concat([s1, s2], axis=1, keys=["A", "B"]))

outer:


,x,y
a,1.0,NaN
b,2.0,10.0
c,3.0,NaN
d,NaN,20.0


inner:


,x,y
b,2,10


keys:


,A,B
a,1.0,NaN
b,2.0,10.0
c,3.0,NaN
d,NaN,20.0


### 4.10 Wydajność: nie sklejaj w pętli

Każde `pd.concat` tworzy nowy obiekt i kopiuje dane → sklejanie w pętli ma złożoność **O(n²)**. Zbieraj kawałki do listy i sklejaj **raz**.

In [29]:
# ŹLE (antywzorzec):
# result = pd.DataFrame()
# for i in range(1000):
#     result = pd.concat([result, chunk_i])     # kopia całości w każdej iteracji

# DOBRZE: lista + jedno concat
parts = [pd.DataFrame({"id": range(i * 3, i * 3 + 3), "chunk": i}) for i in range(4)]
pd.concat(parts, ignore_index=True)

# dla plików: pd.concat((pd.read_csv(p) for p in paths), ignore_index=True)

,id,chunk
0,0,0
1,1,0
2,2,0
3,3,1
4,4,1
5,5,1
6,6,2
7,7,2
8,8,2
9,9,3


### 4.11 Pułapka: zmiana dtype przy brakach

Sklejenie ramek z różnym zestawem kolumn wprowadza `NaN` → kolumny całkowitoliczbowe stają się `float64`. Rozwiązanie: nullable `Int64` po sklejeniu (albo wcześniej).

In [30]:
x = pd.DataFrame({"id": [1, 2], "qty": [5, 6]})
y = pd.DataFrame({"id": [3]})

r = pd.concat([x, y], ignore_index=True)
print(r.dtypes.to_dict())

r["qty"] = r["qty"].astype("Int64")
print(r.dtypes.to_dict())
r

{'id': dtype('int64'), 'qty': dtype('float64')}
{'id': dtype('int64'), 'qty': Int64Dtype()}


,id,qty
0,1,5
1,2,6
2,3,<NA>


## 5. `pd.merge_asof`

```python
pd.merge_asof(left, right, on=None, left_on=None, right_on=None,
              left_index=False, right_index=False, by=None, left_by=None, right_by=None,
              suffixes=("_x", "_y"), tolerance=None, allow_exact_matches=True,
              direction="backward")
```

**Idea:** to jest *left join*, w którym zamiast równości klucza szukamy **najbliższego** klucza w prawej ramce. Dla każdego wiersza lewej ramki wybierany jest **co najwyżej jeden** wiersz z prawej — **nie mnoży** wierszy.

Typowe zastosowania w BI: **kurs waluty z dnia faktury**, cennik/stawka **obowiązująca od daty**, zdarzenie dopasowane do ostatniego pomiaru, sesja → ostatnia kampania przed zakupem.

### Parametry

| Parametr | Wymagany? | Domyślnie | Opis |
|---|---|---|---|
| `left` | 🔴 | — | Lewa ramka. **Zachowane są wszystkie jej wiersze.** |
| `right` | 🔴 | — | Prawa ramka (źródło „najbliższych" wartości). |
| `on` | 🟢* | `None` | Kolumna-klucz „najbliższości": liczbowa lub `datetime`, **posortowana rosnąco w obu ramkach**, bez `NaN`, ta sama nazwa po obu stronach. \*W praktyce wskazujesz klucz: `on`, albo `left_on`+`right_on`, albo `left_index`+`right_index`. Bez tego pandas (dziś) sam użyje wspólnej kolumny — nie polegaj na tym. |
| `left_on` | 🟢 | `None` | Kolumna-klucz w lewej ramce, gdy nazwy się różnią. |
| `right_on` | 🟢 | `None` | Kolumna-klucz w prawej ramce. |
| `left_index` | 🟢 | `False` | Użyj indeksu lewej ramki jako klucza (indeks musi być posortowany). |
| `right_index` | 🟢 | `False` | Użyj indeksu prawej ramki jako klucza. |
| `by` | 🟢 | `None` | Kolumna(y) **dopasowane dokładnie (równość) przed** szukaniem najbliższego — „w obrębie grupy" (np. waluta, ticker, klient). **Nie wymaga sortowania** po `by`. |
| `left_by` | 🟢 | `None` | Kolumna(y) grupujące w lewej ramce (gdy nazwy różne od prawej). |
| `right_by` | 🟢 | `None` | Kolumna(y) grupujące w prawej ramce. |
| `suffixes` | 🟢 | `("_x", "_y")` | Sufiksy dla kolidujących kolumn niebędących kluczami. |
| `tolerance` | 🟢 | `None` | Maksymalna dozwolona odległość klucza. `int`/`float` dla liczb, `pd.Timedelta`/`timedelta` dla dat. Dalsze dopasowania są odrzucane (→ `NaN`). |
| `allow_exact_matches` | 🟢 | `True` | `True` → dopuszcza równość klucza (≤ / ≥); `False` → wymaga nierówności ścisłej (< / >). |
| `direction` | 🟢 | `"backward"` | `"backward"` — ostatni wiersz prawej z kluczem **≤** klucz lewej; `"forward"` — pierwszy z kluczem **≥**; `"nearest"` — najbliższy w dowolną stronę. |

**Wymagania wstępne (kończą się błędem, jeśli nie są spełnione):** `on` posortowany rosnąco w obu ramkach; brak `NaN` w kluczu; identyczny dtype klucza po obu stronach (także **rozdzielczość** `datetime64`).

### 5.1 Podstawy — `direction="backward"` (domyślny)

In [31]:
left = pd.DataFrame({"a": [1, 5, 10], "left_val": ["a", "b", "c"]})
right = pd.DataFrame({"a": [1, 2, 3, 6, 7], "right_val": [1, 2, 3, 6, 7]})

# a=1 -> 1 (równość), a=5 -> 3 (ostatni <= 5), a=10 -> 7
pd.merge_asof(left, right, on="a")

,a,left_val,right_val
0,1,a,1
1,5,b,3
2,10,c,7


### 5.2 `direction` — trzy tryby

In [32]:
for direction in ["backward", "forward", "nearest"]:
    print(f"direction={direction!r}")
    display(pd.merge_asof(left, right, on="a", direction=direction))

direction='backward'


,a,left_val,right_val
0,1,a,1
1,5,b,3
2,10,c,7


direction='forward'


,a,left_val,right_val
0,1,a,1.0
1,5,b,6.0
2,10,c,NaN


direction='nearest'


,a,left_val,right_val
0,1,a,1
1,5,b,6
2,10,c,7


### 5.3 `allow_exact_matches` i `tolerance`

- `allow_exact_matches=False`: dla `a=1` nie wolno użyć równego klucza 1, a wcześniejszego brak → `NaN`.
- `tolerance=2`: dla `a=10` najbliższy poprzedni klucz to 7 (odległość 3 > 2) → `NaN`.

In [33]:
print("allow_exact_matches=False:")
display(pd.merge_asof(left, right, on="a", allow_exact_matches=False))

print("tolerance=2:")
display(pd.merge_asof(left, right, on="a", tolerance=2))

allow_exact_matches=False:


,a,left_val,right_val
0,1,a,NaN
1,5,b,3.0
2,10,c,7.0


tolerance=2:

,a,left_val,right_val
0,1,a,1.0
1,5,b,3.0
2,10,c,NaN


### 5.4 Przykład biznesowy: kurs waluty z dnia faktury (`by` + `tolerance`)

- `on="date"` — szukamy kursu z daty **≤ data faktury**,
- `by="currency"` — ale tylko dla **tej samej waluty**,
- `tolerance` — chroni przed użyciem przestarzałego kursu (np. po długiej przerwie w notowaniach).

Obie ramki muszą być posortowane po `on` (nie po `by`).

In [34]:
rates = pd.DataFrame({
    "date": pd.to_datetime(["2026-09-01", "2026-09-02", "2026-09-04", "2026-09-01", "2026-09-03"]),
    "currency": ["EUR", "EUR", "EUR", "USD", "USD"],
    "rate": [4.26, 4.27, 4.29, 3.81, 3.83],
}).sort_values("date")

invoices = pd.DataFrame({
    "invoice_id": [1, 2, 3, 4, 5],
    "date": pd.to_datetime(["2026-09-02", "2026-09-03", "2026-09-05", "2026-09-03", "2026-09-08"]),
    "currency": ["EUR", "EUR", "EUR", "USD", "USD"],
    "amount": [1000, 500, 700, 900, 300],
}).sort_values("date")

# 1) bez tolerance -- faktura 5 (USD, 08.09) dostaje kurs z 03.09 (5 dni starszy!)
res = pd.merge_asof(invoices, rates, on="date", by="currency")
display(res.assign(amount_pln=res["amount"] * res["rate"]))

# 2) tolerance: kurs starszy niż 3 dni jest odrzucany -> NaN (do obsłużenia / alarmu jakości danych)
pd.merge_asof(invoices, rates, on="date", by="currency", tolerance=pd.Timedelta("3D"))

,invoice_id,date,currency,amount,rate,amount_pln
0,1,2026-09-02,EUR,1000,4.27,4270.0
1,2,2026-09-03,EUR,500,4.27,2135.0
2,4,2026-09-03,USD,900,3.83,3447.0
3,3,2026-09-05,EUR,700,4.29,3003.0
4,5,2026-09-08,USD,300,3.83,1149.0


,invoice_id,date,currency,amount,rate
0,1,2026-09-02,EUR,1000,4.27
1,2,2026-09-03,EUR,500,4.27
2,4,2026-09-03,USD,900,3.83
3,3,2026-09-05,EUR,700,4.29
4,5,2026-09-08,USD,300,NaN


**Audyt: z jakiej daty faktycznie pochodzi kurs?** Kolumna klucza `date` po stronie prawej jest „zjadana" przez `on`. Żeby zachować datę dopasowanego kursu, zduplikuj ją do osobnej kolumny przed złączeniem.

In [35]:
rates_audit = rates.assign(rate_date=rates["date"])

res = pd.merge_asof(invoices, rates_audit, on="date", by="currency", tolerance=pd.Timedelta("3D"))
res["age_days"] = (res["date"] - res["rate_date"]).dt.days
res

,invoice_id,date,currency,amount,rate,rate_date,age_days
0,1,2026-09-02,EUR,1000,4.27,2026-09-02,0.0
1,2,2026-09-03,EUR,500,4.27,2026-09-02,1.0
2,4,2026-09-03,USD,900,3.83,2026-09-03,0.0
3,3,2026-09-05,EUR,700,4.29,2026-09-04,1.0
4,5,2026-09-08,USD,300,NaN,NaT,NaN


### 5.5 `allow_exact_matches=False` — reguła „kurs z dnia poprzedzającego"

Gdy przepisy/reguły biznesowe wymagają kursu z dnia **wcześniejszego** niż data dokumentu, `allow_exact_matches=False` wymusza nierówność ścisłą (`<`).

In [36]:
pd.merge_asof(invoices, rates_audit, on="date", by="currency", allow_exact_matches=False)

,invoice_id,date,currency,amount,rate,rate_date
0,1,2026-09-02,EUR,1000,4.26,2026-09-01
1,2,2026-09-03,EUR,500,4.27,2026-09-02
2,4,2026-09-03,USD,900,3.81,2026-09-01
3,3,2026-09-05,EUR,700,4.29,2026-09-04
4,5,2026-09-08,USD,300,3.83,2026-09-03


### 5.6 `left_on` / `right_on` / `left_by` / `right_by` — różne nazwy kolumn

In [37]:
rates_pl = rates.rename(columns={"date": "data_kursu", "currency": "waluta"})

pd.merge_asof(
    invoices,
    rates_pl,
    left_on="date", right_on="data_kursu",
    left_by="currency", right_by="waluta",
)

,invoice_id,date,currency,amount,data_kursu,waluta,rate
0,1,2026-09-02,EUR,1000,2026-09-02,EUR,4.27
1,2,2026-09-03,EUR,500,2026-09-02,EUR,4.27
2,4,2026-09-03,USD,900,2026-09-03,USD,3.83
3,3,2026-09-05,EUR,700,2026-09-04,EUR,4.29
4,5,2026-09-08,USD,300,2026-09-03,USD,3.83


### 5.7 Klucz w indeksie (`left_index` / `right_index`)

In [38]:
pd.merge_asof(left.set_index("a"), right.set_index("a"), left_index=True, right_index=True)

,left_val,right_val
a,,
1,a,1
5,b,3
10,c,7


### 5.8 Pułapki i komunikaty błędów

1. **Brak sortowania po `on`** → `ValueError: left keys must be sorted` (sortuj po `on`; `by` sortować nie trzeba).
2. **`NaN` w kluczu** → `ValueError: Merge keys contain null values`.
3. **Różny dtype klucza** — także różna **rozdzielczość** czasu (`datetime64[us]` vs `datetime64[ns]`) → `MergeError`. Realne ryzyko, gdy jedna ramka pochodzi z `pd.to_datetime` (w pandas 3.0 wnioskowanie rozdzielczości), a druga z Parquet / bazy danych (np. `ns`) — po stronie źródła bywa inaczej. Ujednolicaj jawnie przez `astype`.

In [39]:
# 1) brak sortowania
try:
    pd.merge_asof(left.iloc[::-1], right, on="a")
except ValueError as e:
    print("ValueError:", e)

# 2) NaN w kluczu
try:
    pd.merge_asof(pd.DataFrame({"a": [1.0, np.nan]}), pd.DataFrame({"a": [1.0, 2.0], "v": [1, 2]}), on="a")
except ValueError as e:
    print("ValueError:", e)

# 3) różna rozdzielczość czasu
ta = pd.DataFrame({"t": pd.to_datetime(["2026-09-01", "2026-09-05"]), "x": [1, 2]})
tb = pd.DataFrame({"t": pd.to_datetime(["2026-09-02"]).astype("datetime64[ns]"), "y": [9]})
print(ta["t"].dtype, "vs", tb["t"].dtype)
try:
    pd.merge_asof(ta, tb, on="t")
except ValueError as e:                       # MergeError dziedziczy po ValueError
    print(type(e).__name__, ":", e)

# naprawa: ujednolicenie dtype
pd.merge_asof(ta, tb.astype({"t": ta["t"].dtype}), on="t")

ValueError: left keys must be sorted
ValueError: Merge keys contain null values on left side
datetime64[us] vs datetime64[ns]
MergeError : incompatible merge keys [0] dtype('<M8[us]') and dtype('<M8[ns]'), must be the same type


,t,x,y
0,2026-09-01,1,NaN
1,2026-09-05,2,9.0


### `merge_asof` — kiedy NIE
- Potrzebujesz **wszystkich** dopasowań w przedziale (np. zdarzenia w oknie ±1 h) → `merge` + filtr (albo range join w DuckDB/Polars).
- Cennik w postaci przedziałów `valid_from`–`valid_to` z **dziurami** → `merge_asof` wybierze „ostatnią obowiązującą od", ale nie sprawdzi `valid_to`; sprawdź go po złączeniu (`res[res["date"] <= res["valid_to"]]`) albo użyj `tolerance`.

## 6. `pd.merge_ordered`

```python
pd.merge_ordered(left, right, on=None, left_on=None, right_on=None,
                 left_by=None, right_by=None, fill_method=None,
                 suffixes=("_x", "_y"), how="outer")
```

**Idea:** złączenie po kluczu, w którym **wynik jest uporządkowany po kluczu**, a luki można od razu wypełnić w przód (`ffill`). Dla szeregów czasowych o różnych częstotliwościach próbkowania.

⚠️ To **nie** jest „najbliższy klucz" jak w `merge_asof`: klucze muszą być **równe** — nierówne dają osobne wiersze (z `NaN` po drugiej stronie).

### Parametry

| Parametr | Wymagany? | Domyślnie | Opis |
|---|---|---|---|
| `left` | 🔴 | — | Lewa ramka (lub `Series` z nazwą). |
| `right` | 🔴 | — | Prawa ramka. |
| `on` | 🟢 | `None` | Kolumna(y) klucza o tej samej nazwie w obu ramkach. Wynik jest **posortowany** po tym kluczu. Dane wejściowe nie muszą być posortowane. |
| `left_on` | 🟢 | `None` | Klucz w lewej ramce (gdy nazwy się różnią). |
| `right_on` | 🟢 | `None` | Klucz w prawej ramce. |
| `left_by` | 🟢 | `None` | Kolumna(y) grupujące **lewą** ramkę — każda grupa jest łączona z `right` osobno (i osobno wypełniana). Musi być `None`, jeśli któraś ramka to `Series`. |
| `right_by` | 🟢 | `None` | Analogicznie dla **prawej** ramki. |
| `fill_method` | 🟢 | `None` | `None` lub `"ffill"` — wypełnianie **w przód** braków powstałych po złączeniu (dotyczy kolumn z obu stron). Inne metody nie są obsługiwane. |
| `suffixes` | 🟢 | `("_x", "_y")` | Sufiksy dla kolidujących kolumn niebędących kluczami (jedna z wartości może być `None`). |
| `how` | 🟢 | `"outer"` | `"outer"`, `"inner"`, `"left"`, `"right"`. ⚠️ Domyślnie **`outer`** (w `merge` — `inner`). |

### 6.1 Podstawy — outer + sortowanie po kluczu

In [40]:
stock = pd.DataFrame({"d": [3, 1, 5], "stock": [30, 10, 50]})     # celowo nieposortowane
price = pd.DataFrame({"d": [4, 2],    "price": [400, 200]})

pd.merge_ordered(stock, price, on="d")

,d,stock,price
0,1,10.0,NaN
1,2,NaN,200.0
2,3,30.0,NaN
3,4,NaN,400.0
4,5,50.0,NaN


### 6.2 `fill_method="ffill"`

Braki po złączeniu są wypełniane **wartością poprzedniego wiersza** — w **obu** kolumnach (pierwsze wiersze, dla których nie ma wcześniejszej wartości, zostają `NaN`).

In [41]:
pd.merge_ordered(stock, price, on="d", fill_method="ffill")

,d,stock,price
0,1,10,NaN
1,2,10,200.0
2,3,30,200.0
3,4,30,400.0
4,5,50,400.0


### 6.3 `how`

In [42]:
for how in ["outer", "inner", "left", "right"]:
    print(f"how={how!r}")
    display(pd.merge_ordered(stock, price, on="d", how=how))

how='outer'


,d,stock,price
0,1,10.0,NaN
1,2,NaN,200.0
2,3,30.0,NaN
3,4,NaN,400.0
4,5,50.0,NaN


how='inner'


,d,stock,price


how='left'


,d,stock,price
0,1,10,NaN
1,3,30,NaN
2,5,50,NaN


how='right'


,d,stock,price
0,2,NaN,200
1,4,NaN,400


### 6.4 `left_by` — wypełnianie **osobno w każdej grupie**

Bez `left_by` `ffill` „przeciekłby" wartość z jednej grupy (np. sklepu) do następnej. Z `left_by` każda grupa jest łączona i wypełniana niezależnie.

**Zastosowanie w BI:** uzupełnianie kalendarza — snapshoty stanów magazynowych per sklep zestawione z pełną tabelą dat.

In [43]:
inventory = pd.DataFrame({
    "store": ["A", "A", "B", "B"],
    "date": pd.to_datetime(["2026-09-01", "2026-09-04", "2026-09-02", "2026-09-05"]),
    "stock": [100, 80, 40, 30],
})
calendar = pd.DataFrame({"date": pd.date_range("2026-09-01", "2026-09-05")})

# każdy sklep osobno: outer z kalendarzem + ffill w obrębie sklepu
pd.merge_ordered(inventory, calendar, on="date", left_by="store", fill_method="ffill")

,store,date,stock
0,A,2026-09-01,100.0
1,A,2026-09-02,100.0
2,A,2026-09-03,100.0
3,A,2026-09-04,80.0
4,A,2026-09-05,80.0
5,B,2026-09-01,NaN
6,B,2026-09-02,40.0
7,B,2026-09-03,40.0
8,B,2026-09-04,40.0
9,B,2026-09-05,30.0


Zwróć uwagę: dla sklepu **B** dzień `2026-09-01` (przed pierwszym snapshotem) ma `NaN` — nie ma czego wypełnić w przód.

### 6.5 Równoważnik „ręczny"

`merge_ordered(how="outer", fill_method="ffill")` daje te same **wartości** co `merge(how="outer")` + `sort_values` + `ffill` (mogą różnić się dtypes). Wersja ręczna jest bardziej **jawna** i pozwala wypełniać wybrane kolumny; `merge_ordered` jest zwięzła i ma wbudowane `left_by`/`right_by`.

In [44]:
manual = (
    stock.merge(price, on="d", how="outer")
         .sort_values("d")
         .ffill()
         .reset_index(drop=True)
)
auto = pd.merge_ordered(stock, price, on="d", fill_method="ffill")

# wartości są takie same; różni się tylko dtype (ręczny ffill zostawia float, merge_ordered może zachować int)
pd.testing.assert_frame_equal(manual, auto, check_dtype=False)
print("wartości identyczne; dtypes:", manual["stock"].dtype, "vs", auto["stock"].dtype)
manual

wartości identyczne; dtypes: float64 vs int64


,d,stock,price
0,1,10.0,NaN
1,2,10.0,200.0
2,3,30.0,200.0
3,4,30.0,400.0
4,5,50.0,400.0


### `merge_asof` vs `merge_ordered` vs `merge`

| Cecha | `merge` | `merge_asof` | `merge_ordered` |
|---|---|---|---|
| Dopasowanie klucza | równość | **najbliższy** (≤, ≥, nearest) | równość |
| Domyślny `how` | `inner` | (zawsze left) | `outer` |
| Mnożenie wierszy | tak (n:m) | **nie** (max 1 dopasowanie) | tak |
| Wymaga sortowania wejścia | nie | **tak** (`on`) | nie (wynik i tak sortowany) |
| Wypełnianie luk | nie | wynika z natury (najbliższy) | `fill_method="ffill"` |
| Grupowanie | — | `by` (równość) | `left_by` / `right_by` (osobne złączenie per grupa) |
| Typowy przypadek | JOIN wymiar↔fakt | kurs/cennik/pomiar „obowiązujący w dniu" | scalenie szeregów, kalendarz + snapshoty |

## 7. Częściowo nakładające się dane: `combine_first`, `combine`, `update`

To trzy narzędzia do sytuacji, gdy **te same „komórki"** (ten sam wiersz i ta sama kolumna) mogą występować w obu ramkach i trzeba zdecydować, którą wartość zachować. Kluczowa różnica względem `merge`:

> `combine_first` / `combine` / `update` dopasowują po **etykietach indeksu i kolumn**, a nie po kolumnie-kluczu. Żeby uzupełniać dane po kluczu biznesowym (np. `customer_id`), najpierw ustaw go jako indeks: `set_index("customer_id")`.

| | `combine_first` | `combine` | `update` |
|---|---|---|---|
| Rola | „uzupełnij braki z drugiej ramki" | „wybierz wartość funkcją" | „nadpisz wartościami z drugiej ramki" |
| Zwraca | **nową** ramkę | **nową** ramkę | `None` — modyfikuje **w miejscu** |
| Indeks/kolumny wyniku | **suma** obu | **suma** obu | tylko ramki wywołującej |
| Priorytet | `self`, braki z `other` | zależy od `func` | `other` nadpisuje `self` (poza `NaN` z `other`) |

### 7.1 `combine_first`

```python
DataFrame.combine_first(other)      # także Series.combine_first(other)
```

| Parametr | Wymagany? | Domyślnie | Opis |
|---|---|---|---|
| `other` | 🔴 | — | Ramka (dla `Series.combine_first` — seria), z której pochodzą wartości uzupełniające. To **jedyny** parametr. |

Zasada: wartość z `self` jest zachowana, **jeśli nie jest null**; w przeciwnym razie brana jest wartość z `other` (o tej samej etykiecie wiersza i kolumny). Wynik ma **sumę** etykiet indeksu i kolumn.

**Różnica względem `fillna(other)`:** `fillna` zachowuje kształt `self` (nie dodaje wierszy/kolumn z `other`), `combine_first` — rozszerza do sumy.

In [45]:
d1 = pd.DataFrame({"A": [1, np.nan], "B": [np.nan, 4]}, index=[0, 1])
d2 = pd.DataFrame({"A": [9, 9, 9], "C": [5, 5, 5]}, index=[0, 1, 2])

print("combine_first -- suma indeksów (0,1,2) i kolumn (A,B,C):")
display(d1.combine_first(d2))

print("fillna(d2) -- zachowuje kształt d1:")
display(d1.fillna(d2))

combine_first -- suma indeksów (0,1,2) i kolumn (A,B,C):


,A,B,C
0,1.0,NaN,5
1,9.0,4.0,5
2,9.0,NaN,5


fillna(d2) -- zachowuje kształt d1:


,A,B
0,1.0,NaN
1,9.0,4.0


#### Przykład biznesowy: dwa źródła danych o klientach (CRM ma priorytet, ERP uzupełnia braki)

Dopasowanie idzie po indeksie, więc klucz biznesowy ustawiamy przez `set_index`. Klient `4` istnieje tylko w ERP — trafia do wyniku (suma indeksów).

Wielu źródeł: łańcuch `a.combine_first(b).combine_first(c)` — priorytet malejący od lewej.

In [46]:
crm = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "email": ["anna@crm.pl", None, "celina@crm.pl"],
    "phone": [None, "500-000-002", None],
})
erp = pd.DataFrame({
    "customer_id": [2, 3, 4],
    "email": ["bartek@erp.pl", "celina@erp.pl", "dawid@erp.pl"],
    "phone": ["600-000-002", "600-000-003", "600-000-004"],
})

merged = (
    crm.set_index("customer_id")
       .combine_first(erp.set_index("customer_id"))
       .reset_index()
)
merged

,customer_id,email,phone
0,1,anna@crm.pl,NaN
1,2,bartek@erp.pl,500-000-002
2,3,celina@crm.pl,600-000-003
3,4,dawid@erp.pl,600-000-004


Zauważ: e-mail Anny (`crm`) i Celiny (`crm`) zostały zachowane mimo że ERP też je ma; braki Bartka (e-mail) i Celiny (telefon) uzupełnione z ERP.

**Uwaga na dtype:** po `combine_first` typy mogą się zmienić (np. `int` → `float` przy `NaN` po rozszerzeniu indeksu). Sprawdź `.dtypes` i w razie potrzeby `astype("Int64")`.

### 7.2 `combine`

```python
DataFrame.combine(other, func, fill_value=None, overwrite=True)     # także Series.combine(other, func, fill_value=None)
```

| Parametr | Wymagany? | Domyślnie | Opis |
|---|---|---|---|
| `other` | 🔴 | — | Ramka do połączenia kolumna po kolumnie. |
| `func` | 🔴 | — | Funkcja `f(s1, s2)`, przyjmująca **dwie serie (kolumny o tej samej nazwie)** i zwracająca serię lub skalar. Wywoływana **raz na kolumnę**, nie na komórkę — dla działania „element po elemencie" użyj funkcji wektorowej (`np.minimum`, `Series.where` …). |
| `fill_value` | 🟢 | `None` | Wartość, którą wypełniane są `NaN` **przed** przekazaniem kolumn do `func`. |
| `overwrite` | 🟢 | `True` | `True` — kolumny `self`, których nie ma w `other`, są nadpisywane `NaN`; `False` — zostają zachowane. |

Wynik ma sumę indeksów i kolumn obu ramek. Dla `Series.combine`: `func` wywoływana jest **na parze skalarów** (element po elemencie), a parametrów `overwrite` nie ma.

#### `func` wektorowa — element po elemencie

In [47]:
c1 = pd.DataFrame({"A": [5, 0], "B": [2, 4]})
c2 = pd.DataFrame({"A": [1, 1], "B": [3, 3]})

# dla każdej komórki bierzemy mniejszą wartość
c1.combine(c2, np.minimum)

,A,B
0,1,2
1,0,3


#### `func` decydująca na poziomie całej kolumny

Poniższa funkcja dla każdej kolumny wybiera **tę z dwóch**, która ma mniejszą sumę — decyzja zapada na poziomie kolumny, nie komórki.

In [48]:
d_a = pd.DataFrame({"A": [0, 0], "B": [4, 4]})
d_b = pd.DataFrame({"A": [1, 1], "B": [3, 3]})

take_smaller = lambda s1, s2: s1 if s1.sum() < s2.sum() else s2
d_a.combine(d_b, take_smaller)

,A,B
0,0,3
1,0,3


#### `fill_value` — wypełnienie `NaN` przed wywołaniem `func`

In [49]:
e1 = pd.DataFrame({"A": [0, 0], "B": [None, 4]})
e2 = pd.DataFrame({"A": [1, 1], "B": [3, 3]})

print("bez fill_value (NaN trafia do func):")
display(e1.combine(e2, np.minimum))
print("fill_value=-5 (NaN -> -5 przed func):")
display(e1.combine(e2, np.minimum, fill_value=-5))

bez fill_value (NaN trafia do func):


,A,B
0,0,NaN
1,0,3.0


fill_value=-5 (NaN -> -5 przed func):


,A,B
0,0,-5.0
1,0,3.0


#### `overwrite` i różne osie

Gdy ramki mają różne kolumny/indeksy, wynik ma ich sumę. `overwrite=True` zeruje (`NaN`) kolumny `self`, których brak w `other`; `overwrite=False` je zachowuje.

In [50]:
f1 = pd.DataFrame({"A": [0, 0], "B": [4, 4]})
f2 = pd.DataFrame({"B": [3, 3], "C": [-10, 1]}, index=[1, 2])

print("overwrite=True (domyślnie):")
display(f1.combine(f2, np.minimum))
print("overwrite=False:")
display(f1.combine(f2, np.minimum, overwrite=False))

overwrite=True (domyślnie):


,A,B,C
0,NaN,NaN,NaN
1,NaN,3.0,NaN
2,NaN,NaN,NaN


overwrite=False:


,A,B,C
0,0.0,NaN,NaN
1,0.0,3.0,NaN
2,NaN,NaN,NaN


#### Uwaga o wydajności i lepsze alternatywy

`combine` wywołuje `func` per kolumna w pętli Pythona po kolumnach i przechodzi przez wyrównanie indeksów — dla prostych operacji istnieją szybsze, jawne odpowiedniki:

| Zamiast | Użyj |
|---|---|
| `a.combine(b, lambda x, y: x + y, fill_value=0)` | `a.add(b, fill_value=0)` (także `sub`, `mul`, `div`) |
| `a.combine(b, np.minimum)` | `np.minimum(a, b)` po `a.align(b)` lub `a.where(a < b, b)` |
| `a.combine(b, lambda x, y: x.where(x.notna(), y))` | `a.combine_first(b)` |

`combine` ma sens dla **niestandardowej reguły decyzyjnej** na całej kolumnie, której nie da się wyrazić operatorem.

In [51]:
g1 = pd.DataFrame({"A": [1, 2], "B": [3, np.nan]}, index=["x", "y"])
g2 = pd.DataFrame({"A": [10, 20], "B": [30, 40]}, index=["y", "z"])

print("combine z lambdą:")
display(g1.combine(g2, lambda s1, s2: s1 + s2, fill_value=0))
print("add(fill_value=0) -- to samo, szybciej i czytelniej:")
display(g1.add(g2, fill_value=0))

combine z lambdą:


,A,B
x,1.0,3.0
y,12.0,30.0
z,20.0,40.0


add(fill_value=0) -- to samo, szybciej i czytelniej:


,A,B
x,1.0,3.0
y,12.0,30.0
z,20.0,40.0


### 7.3 `update` (pokrewne, modyfikuje w miejscu)

```python
DataFrame.update(other, join="left", overwrite=True, filter_func=None, errors="ignore")   # zwraca None
```

| Parametr | Wymagany? | Domyślnie | Opis |
|---|---|---|---|
| `other` | 🔴 | — | Ramka (lub `Series` z nazwą) z nowymi wartościami; dopasowanie po etykietach indeksu i kolumn. |
| `join` | 🟢 | `"left"` | Jedyna obsługiwana wartość: `"left"` — kształt wynikowy = kształt `self` (nowe wiersze/kolumny z `other` **nie są** dodawane). |
| `overwrite` | 🟢 | `True` | `True` — nadpisuje wartości `self` wartościami `other` (poza `NaN` w `other`); `False` — uzupełnia **tylko braki** w `self`. |
| `filter_func` | 🟢 | `None` | `callable` przyjmujący tablicę 1-D **wartości z `self`** i zwracający tablicę bool: `True` = ta wartość **może zostać zaktualizowana**. |
| `errors` | 🟢 | `"ignore"` | `"raise"` → `ValueError`, jeśli obie ramki mają wartości nie-`NaN` w tej samej komórce (kontrola konfliktów). |

**Kluczowe własności:** wartości `NaN` w `other` **nigdy** nie nadpisują niczego; metoda nie zwraca wyniku (modyfikuje `self`).

⚠️ **Copy-on-Write (pandas 3.0):** `df["A"].update(...)` **nie zadziała** (modyfikuje kopię) — użyj `df.update(...)` na całej ramce (albo `df.update({"A": ...})`).

In [52]:
base = pd.DataFrame({"A": [1, 2, 3], "B": [10.0, np.nan, 30.0]})
upd  = pd.DataFrame({"A": [np.nan, 20, 30], "B": [np.nan, 25, np.nan]})

x = base.copy(); x.update(upd)
print("domyślnie (overwrite=True) -- NaN z upd nic nie nadpisują:")
display(x)

x = base.copy(); x.update(upd, overwrite=False)
print("overwrite=False -- tylko braki w base:")
display(x)

x = base.copy(); x.update(upd, filter_func=lambda vals: vals < 3)
print("filter_func -- aktualizuj tylko wartości < 3 w base (B[1] zostaje NaN: NaN < 3 -> False):")
display(x)

x = base.copy()
try:
    x.update(upd, errors="raise")          # A[1], A[2] mają wartości po obu stronach
except ValueError as e:
    print("ValueError:", e)

domyślnie (overwrite=True) -- NaN z upd nic nie nadpisują:


,A,B
0,1,10.0
1,20,25.0
2,30,30.0


overwrite=False -- tylko braki w base:


,A,B
0,1,10.0
1,2,25.0
2,3,30.0


filter_func -- aktualizuj tylko wartości < 3 w base (B[1] zostaje NaN: NaN < 3 -> False):


,A,B
0,1,10.0
1,20,NaN
2,3,30.0


ValueError: Data overlaps.


### Który wybrać?

- **Koalescencja źródeł** (pierwsza niepusta wartość z kolejnych źródeł) → `combine_first` (albo `Series.combine_first`); w Polars/SQL: `pl.coalesce(...)` / `COALESCE(...)`.
- **Aktualizacja tabeli wartościami z korekty**, z kontrolą konfliktów → `update(..., errors="raise")`.
- **Własna logika wyboru** (min/max/priorytet warunkowy) → `combine` z funkcją wektorową; przy prostej arytmetyce → `add/sub/mul/div(fill_value=...)`.
- **Dopasowanie po kluczu biznesowym z dołożeniem kolumn** → nie ta rodzina; użyj `merge`.

## 8. Wydajność, dobre praktyki, alternatywy

### Dobre praktyki (pandas)

1. **Ujednolicaj dtypes kluczy** przed złączeniem (int vs str, rozdzielczość datetime). Klucze `category`/integer są szybsze niż tekstowe.
2. **Redukuj dane przed złączeniem** — tylko potrzebne kolumny i wiersze (`right[["key", "col"]]`), agreguj po stronie „wiele" *przed* JOIN-em.
3. **Zawsze deklaruj oczekiwaną kardynalność** (`validate="m:1"`) w pipeline’ach — cichy wybuch wierszy to najczęstszy błąd.
4. **Audytuj** (`indicator=True`) albo sprawdzaj liczność wyniku (`assert len(res) == len(left)`) po `left join`.
5. **`concat` raz**, nie w pętli; przy dużej liczbie plików — generator do `pd.concat`.
6. Dla `merge_asof` **sortuj po `on`** i trzymaj rozdzielczość czasu spójną.
7. Anti/semi-join: `isin` / `~isin`, nie `merge` + filtr.

### Podział odpowiedzialności w stacku

| Operacja | Najlepsze miejsce | Uwagi |
|---|---|---|
| JOIN tabel faktów/wymiarów | **SQL Server** (po stronie bazy) | Filtrowanie i agregacja przed przesłaniem danych; indeksy na kluczach. |
| „Najbliższy" klucz (as-of) | **SQL Server**: `OUTER APPLY (SELECT TOP 1 ... ORDER BY ...)` albo Python (`merge_asof` / DuckDB `ASOF JOIN` / Polars `join_asof`) | Power Query nie ma natywnego as-of join; w DAX wymaga własnych wzorców (np. `CALCULATE` z filtrem po `MAX` daty) — kosztowne przy dużych tabelach, lepiej policzyć wcześniej w SQL/Pythonie. |
| Łączenie w Power Query | `Table.NestedJoin` / `Merge Queries` — zwykle **zachowuje query folding**, gdy oba źródła są w tej samej bazie i poprzednie kroki też się „foldują" | Złączenia po stronie serwera są zwykle wielokrotnie szybsze niż w silniku M. |
| Duże zbiory w Pythonie | **Polars / DuckDB** | Wielowątkowość, lazy evaluation, mniejsze zużycie pamięci. |
| Relacje w modelu | **Power BI** (model gwiazdy, relacje 1:n) | Zamiast fizycznego łączenia tabel w M/SQL, jeśli nie trzeba spłaszczać. |

### SQL Server — as-of join (nie wykonywany w notebooku)

```sql
-- kurs waluty z dnia faktury (backward), jeden kurs na fakturę
SELECT i.invoice_id, i.invoice_date, i.currency, i.amount, r.rate, r.rate_date
FROM   dbo.Invoices AS i
OUTER APPLY (
    SELECT TOP (1) r.rate, r.rate_date
    FROM   dbo.Rates AS r
    WHERE  r.currency  = i.currency
      AND  r.rate_date <= i.invoice_date          -- '<' dla reguły "dzień poprzedzający"
    ORDER BY r.rate_date DESC
) AS r;
-- indeks wspierający: (currency, rate_date DESC) INCLUDE (rate)
```

### Polars — `join_asof`, `join(how="anti")`, `coalesce`

Wymagania jak w pandas: klucz `on` posortowany. `strategy` odpowiada `direction`, `by` — `by`, `tolerance` przyjmuje np. `"3d"`, `allow_exact_matches` działa tak samo.

In [53]:
import polars as pl

inv_pl = pl.from_pandas(invoices).sort("date")
rates_pl_df = pl.from_pandas(rates).sort("date").with_columns(rate_date=pl.col("date"))

# as-of join: kurs <= data faktury, per waluta, tolerancja 3 dni
inv_pl.join_asof(
    rates_pl_df,
    on="date", by="currency",
    strategy="backward",      # = direction="backward"
    tolerance="3d",           # = tolerance=pd.Timedelta("3D")
    check_sortedness=False,   # Polars i tak nie sprawdza sortowania przy `by` (ostrzeżenie) -- sortujemy ręcznie wyżej
)

invoice_id,date,currency,amount,rate,rate_date
i64,datetime[μs],str,i64,f64,datetime[μs]
1,2026-09-02 00:00:00,"""EUR""",1000,4.27,2026-09-02 00:00:00
2,2026-09-03 00:00:00,"""EUR""",500,4.27,2026-09-02 00:00:00
4,2026-09-03 00:00:00,"""USD""",900,3.83,2026-09-03 00:00:00
3,2026-09-05 00:00:00,"""EUR""",700,4.29,2026-09-04 00:00:00
5,2026-09-08 00:00:00,"""USD""",300,null,null


In [54]:
# anti-join i semi-join w Polars
cust_pl = pl.from_pandas(customers)
ord_pl = pl.from_pandas(orders)

print("anti (klienci bez zamówień):")
display(cust_pl.join(ord_pl, on="customer_id", how="anti"))
print("semi (klienci z zamówieniami, bez mnożenia wierszy):")
display(cust_pl.join(ord_pl, on="customer_id", how="semi"))

# combine_first -> pl.coalesce: pierwsza niepusta wartość
src = pl.DataFrame({"a": [1, None, None], "b": [None, 2, None], "c": [9, 9, 3]})
src.with_columns(first_value=pl.coalesce("a", "b", "c"))

anti (klienci bez zamówień):


customer_id,name,city
i64,str,str
4,"""Dawid""","""Poznań"""


semi (klienci z zamówieniami, bez mnożenia wierszy):


customer_id,name,city
i64,str,str
1,"""Anna""","""Bydgoszcz"""
2,"""Bartek""","""Toruń"""
3,"""Celina""","""Gdańsk"""


a,b,c,first_value
i64,i64,i64,i64
1,null,9,1
null,2,9,2
null,null,3,3


### DuckDB — `ASOF JOIN` bezpośrednio na ramkach pandas

DuckDB widzi zmienne pandas/Polars po nazwie (replacement scan), więc nie trzeba niczego kopiować ani ładować. `ASOF JOIN` **nie wymaga wcześniejszego sortowania** — silnik zajmuje się tym sam.

- `i.date >= r.date` ⇔ `direction="backward"`, `allow_exact_matches=True`; z `>` → `allow_exact_matches=False`; z `<=` → `direction="forward"`.
- Warunek równości (`i.currency = r.currency`) ⇔ `by`.

In [55]:
import duckdb

duckdb.sql("""
    SELECT i.invoice_id, i.date, i.currency, i.amount, r.rate, r.rate_date
    FROM invoices AS i
    ASOF LEFT JOIN rates_audit AS r
      ON i.currency = r.currency
     AND i.date >= r.rate_date
    ORDER BY i.invoice_id
""").df()

,invoice_id,date,currency,amount,rate,rate_date
0,1,2026-09-02,EUR,1000,4.27,2026-09-02
1,2,2026-09-03,EUR,500,4.27,2026-09-02
2,3,2026-09-05,EUR,700,4.29,2026-09-04
3,4,2026-09-03,USD,900,3.83,2026-09-03
4,5,2026-09-08,USD,300,3.83,2026-09-03


In [56]:
# anti-join i uzupełnianie braków (COALESCE) w DuckDB
print("ANTI JOIN:")
display(duckdb.sql("""
    SELECT * FROM customers c
    ANTI JOIN orders o ON c.customer_id = o.customer_id
""").df())

print("COALESCE zamiast combine_first (priorytet: crm, potem erp):")
duckdb.sql("""
    SELECT COALESCE(c.customer_id, e.customer_id) AS customer_id,
           COALESCE(c.email, e.email)             AS email,
           COALESCE(c.phone, e.phone)             AS phone
    FROM crm c
    FULL OUTER JOIN erp e USING (customer_id)
    ORDER BY 1
""").df()

ANTI JOIN:


,customer_id,name,city
0,4,Dawid,Poznań


COALESCE zamiast combine_first (priorytet: crm, potem erp):


,customer_id,email,phone
0,1,anna@crm.pl,NaN
1,2,bartek@erp.pl,500-000-002
2,3,celina@crm.pl,600-000-003
3,4,dawid@erp.pl,600-000-004


**`merge_ordered` w Polars/DuckDB:** Polars — `left.join(right, on="d", how="full", coalesce=True).sort("d")` + `fill_null(strategy="forward")`; DuckDB — `FULL OUTER JOIN … ORDER BY` + `LAST_VALUE(col IGNORE NULLS) OVER (ORDER BY d)`. Przy grupach (odpowiednik `left_by`) dodaj `.over("store")` (Polars) lub `PARTITION BY store` (DuckDB).

### Kiedy zostać przy pandas, a kiedy zmienić narzędzie

- Dane mieszczą się wygodnie w RAM, złączenie jednorazowe, eksploracja → **pandas**.
- Wielomilionowe tabele, powtarzalny ETL, kilka JOIN-ów pod rząd → **DuckDB** (SQL) lub **Polars** (lazy API).
- Dane już w SQL Server i wynik wraca do Power BI → **SQL po stronie bazy** (mniej danych po sieci, indeksy, brak własnej kopii w pamięci).

## 9. Ściąga

| Funkcja | Wymagane | Najważniejsze opcjonalne (domyślnie) | Pamiętaj |
|---|---|---|---|
| `merge` | `left`, `right` | `how` (`inner`), `on`/`left_on`/`right_on`, `suffixes`, `indicator` (`False`), `validate` (`None`) | `outer` sortuje klucze; `NaN` łączy się z `NaN`; n:m mnoży wiersze; do filtrowania lepsze `isin` |
| `join` | `other` | `on`, `how` (**`left`**), `lsuffix`/`rsuffix` (`""`) | domyślnie po indeksie; kolizja nazw bez sufiksów → błąd; lista ramek OK |
| `concat` | `objs` | `axis` (`0`), `join` (`outer`), `ignore_index`, `keys`, `names`, `verify_integrity`, `sort` | wszystko poza `objs` keyword-only; nie w pętli; `axis=1` = join po indeksie |
| `merge_asof` | `left`, `right` | `on`, `by`, `direction` (`backward`), `tolerance`, `allow_exact_matches` (`True`) | posortowane `on`, bez `NaN`, ten sam dtype (także rozdzielczość czasu); max 1 dopasowanie |
| `merge_ordered` | `left`, `right` | `on`, `how` (**`outer`**), `fill_method` (`None`/`"ffill"`), `left_by`/`right_by` | wynik posortowany po kluczu; `ffill` w obu kolumnach; `left_by` = osobno per grupa |
| `combine_first` | `other` | — | braki z `other`; suma indeksów i kolumn; po **etykietach** (nie po kluczu) |
| `combine` | `other`, `func` | `fill_value` (`None`), `overwrite` (`True`) | `func` per kolumna; proste operacje → `add(fill_value=...)` |
| `update` | `other` | `join` (`"left"`), `overwrite` (`True`), `filter_func`, `errors` (`"ignore"`) | in-place, zwraca `None`; `NaN` z `other` nie nadpisują; nie na `df["col"]` (CoW) |